# Knowledge Graph Construction WITHOUT Coreference Resolution

**Source data:** `hf://datasets/facebook/wiki_dpr/data/psgs_w100/nq/train-00000-of-00157.parquet`

This notebook constructs a knowledge graph using:
- **NO Coreference Resolution** (entities stay as-is, aliases remain separate)
- **REBEL** for relation extraction
- **SpaCy** for dependency parsing and NER
- **NetworkX MultiGraph** for graph representation

**Why 3 parts?**
To ensure a fair comparison with `kg_with_coref.ipynb`, we use the SAME 3 parts.
Coreference resolution is memory-intensive, so the with-coref notebook processes the corpus in 3 chunks.
This notebook loads the same 3 chunks but simply skips the coreference step.

**Pipeline:**
1. Run `prepare_data_splits.ipynb` once → splits parquet into 3 parts grouped by `title`
2. Load part 1 → SKIP coref → extract relations (REBEL + SpaCy) → add to graph
3. Load part 2 → SKIP coref → extract relations → add to graph
4. Load part 3 → SKIP coref → extract relations → add to graph
5. Save final graph

The ONLY difference vs. `kg_with_coref.ipynb` is that steps 2-4 do NOT apply coreference resolution.

In [5]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.5 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [1]:
import re
import json
import math
import unicodedata
import pickle
import glob
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import networkx as nx
import spacy
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


## 1. Configuration

In [16]:
# =============================================================================
# CONFIGURATION
# =============================================================================

REBEL_MODEL_NAME = "Babelscape/rebel-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
MIN_REBEL_CONF = 0.40
MIN_GROUND_SCORE = 0.50
EXPORT_GRAPHML = True
EXPORT_JSON = True

# Valid NER types for entity extraction
VALID_NER = {"PERSON", "ORG", "GPE", "LOC", "WORK_OF_ART", "EVENT", "PRODUCT"}

# Banned pronouns (entities that should not become nodes)
# BAD_PRONOUNS = {
#     "he", "she", "it", "they", "them", "this", "that",
#     "these", "those", "i", "you", "we",
#     "who", "which", "where", "when", "what"
# }

BAD_PRONOUNS = {}

# Blacklist relations (too generic / noisy)
BLACKLIST_RELATIONS = {
    "be", "have", "do", "become", "take",
    "instance of", "has part", "part of",
    "country", "genre", "occupation", "publication date"
}

# Bad objects (too generic to be meaningful nodes)
BAD_OBJECTS = {
    "yes", "no", "okay", "song", "music", "review", "album",
    "time", "demo", "mixture", "schedule", "show", "tour",
    "event", "episode", "records", "campaign", "talks", "lawyer"
}

# REBEL-specific bad relations
BAD_REBEL_RELATIONS = {
    "genre", "instance of", "subclass of",
    "country", "occupation", "publication date"
}

print("Configuration loaded")

Configuration loaded


## 2. Load Models (NO Coreference)

In [6]:
# =============================================================================
# LOAD MODELS (NO Coreference)
# =============================================================================

print("Loading spaCy...")
nlp = spacy.load("en_core_web_sm")
print("Using en_core_web_sm")

# NOTE: We intentionally do NOT load FastCoref here
# Entity aliases (e.g., 'Einstein' vs 'Albert Einstein') remain as separate nodes

print("Loading REBEL...")
rebel_tokenizer = AutoTokenizer.from_pretrained(REBEL_MODEL_NAME)
rebel_model = AutoModelForSeq2SeqLM.from_pretrained(REBEL_MODEL_NAME)
rebel_model.to(DEVICE)
rebel_model.eval()
print("REBEL ready on", DEVICE)

Loading spaCy...
Using en_core_web_sm
Loading REBEL...


2026-07-01 21:07:23.296512: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


REBEL ready on cuda


## 3. Global State

In [7]:
# =============================================================================
# GLOBAL STATE
# =============================================================================

LAST_PERSON = None
LAST_ORG = None
LAST_GPE = None

ENTITY_CACHE = {}
PERSON_ALIASES = {}
ORG_ALIASES = {}

G = nx.MultiGraph()
REL_COUNTER = Counter()
ENTITY_COUNTER = Counter()
EDGE_STORE = {}

print("Global state initialized")

Global state initialized


## 4. Load Pre-Split Data Parts (Same for Both Pipelines)

**Run `prepare_data_splits.ipynb` first** to generate the 3 parts from the single parquet file.

**Why split into 3 parts?**
To ensure a fair comparison, this notebook loads the SAME 3 chunks — it simply skips the coreference step.
Both notebooks load these identical parts so the ONLY difference is whether coreference resolution is applied.

In [8]:
# Load the 3 pre-split parts (generated by prepare_data_splits.py)
# We use the same 3 parts as kg_with_coref.ipynb for a fair comparison
split_files = sorted(glob.glob("./prepared_splits/part_*_of_3.pkl"))

if len(split_files) != 3:
    raise FileNotFoundError(
        "Pre-split parts not found. Run prepare_data_splits.py first to split the parquet."
    )

parts = []
for f in split_files:
    with open(f, "rb") as fh:
        parts.append(pickle.load(fh))
    print(f"Loaded: {f}")

for i, part in enumerate(parts):
    print(f"  Part {i+1}: {len(part):,} rows, columns={part.columns.tolist()}")

print(f"\nTotal rows across all parts: {sum(len(p) for p in parts):,}")
print("\nNOTE: These are the SAME 3 parts used by kg_with_coref.ipynb")
print("NOTE: This notebook skips coreference resolution but processes the same chunks.")

Loaded: ./prepared_splits/part_1_of_3.pkl
Loaded: ./prepared_splits/part_2_of_3.pkl
Loaded: ./prepared_splits/part_3_of_3.pkl
  Part 1: 43,008 rows, columns=['id', 'text', 'title', 'embeddings']
  Part 2: 41,086 rows, columns=['id', 'text', 'title', 'embeddings']
  Part 3: 49,762 rows, columns=['id', 'text', 'title', 'embeddings']

Total rows across all parts: 133,856

NOTE: These are the SAME 3 parts used by kg_with_coref.ipynb
NOTE: This notebook skips coreference resolution but processes the same chunks.


## 5. Entity Utilities

In [9]:
# =============================================================================
# ENTITY UTILITIES
# =============================================================================

def normalize_entity(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()
    text = re.sub(r"<pad>", "", text)
    text = re.sub(r"</?s>", "", text)
    text = re.sub(r"\(.*?\)", "", text)
    text = re.sub(r"\"|\'|`", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\-]", "", text)
    return text.strip()


def update_entity_memory(doc):
    global LAST_PERSON, LAST_ORG, LAST_GPE
    for ent in doc.ents:
        entity = canonicalize_entity(ent.text)
        if ent.label_ == "PERSON":
            LAST_PERSON = entity
            parts = entity.split()
            if len(parts) >= 2:
                PERSON_ALIASES[parts[-1]] = entity
                PERSON_ALIASES[parts[0]] = entity
        elif ent.label_ == "ORG":
            LAST_ORG = entity
            parts = entity.split()
            if len(parts) >= 2:
                ORG_ALIASES[parts[0]] = entity
                ORG_ALIASES[parts[-1]] = entity
        elif ent.label_ in {"GPE", "LOC"}:
            LAST_GPE = entity


def resolve_pronoun(entity):
    entity = normalize_entity(entity)
    if not entity:
        return entity
    if entity in {"he", "him", "his", "she", "her", "hers"}:
        if LAST_PERSON:
            return LAST_PERSON
    if entity in {"it", "its"}:
        if LAST_ORG:
            return LAST_ORG
        if LAST_GPE:
            return LAST_GPE
    if entity in {"they", "them", "their", "theirs"}:
        if LAST_ORG:
            return LAST_ORG
        if LAST_PERSON:
            return LAST_PERSON
    if entity in PERSON_ALIASES:
        return PERSON_ALIASES[entity]
    if entity in ORG_ALIASES:
        return ORG_ALIASES[entity]
    return entity


def canonicalize_entity(entity):
    entity = normalize_entity(entity)
    if not entity:
        return entity
    if entity in ENTITY_CACHE:
        return ENTITY_CACHE[entity]
    # Entity linking disabled (spacy-entity-linker not required)
    ENTITY_CACHE[entity] = entity
    return entity


def clean_entity(text):
    text = resolve_pronoun(text)
    text = canonicalize_entity(text)
    text = normalize_entity(text)
    if not text:
        return None
    if text in BAD_PRONOUNS:
        return None
    if text in BAD_OBJECTS:
        return None
    if len(text.split()) > 20:
        return None
    return text


def entity_span(token):
    doc = token.doc
    for ent in doc.ents:
        if ent.start <= token.i < ent.end:
            return clean_entity(ent.text)
    subtree_tokens = [t.text for t in token.subtree if t.dep_ != "punct"]
    text = " ".join(subtree_tokens)
    return clean_entity(text)


def noisy_sentence(sentence):
    s = sentence.lower()
    lyric_patterns = [
        r"\bi love you\b", r"\byou know\b", r"\bme\b",
        r"\bi\b", r"lyrics", r"chorus"
    ]
    if sentence.count('"') >= 4:
        return True
    for p in lyric_patterns:
        if re.search(p, s):
            return True
    return False


def grounding_score(entity, sentence, ner_entities):
    e = normalize_entity(entity)
    sent = normalize_entity(sentence)
    score = 0.0
    if e in ner_entities:
        score += 0.65
    if e in sent:
        score += 0.25
    entity_tokens = set(e.split())
    sent_tokens = set(sent.split())
    overlap = len(entity_tokens & sent_tokens)
    if entity_tokens:
        score += (overlap / len(entity_tokens)) * 0.10
    return min(score, 1.0)


def validate_triple(triple):
    s, r, o, conf, src = triple
    s = clean_entity(s)
    r = normalize_entity(r)
    o = clean_entity(o)
    if not s or not o or not r:
        return False
    if s == o:
        return False
    if r in BLACKLIST_RELATIONS:
        return False
    if conf <= 0:
        return False
    return True

print("Entity utilities loaded")

Entity utilities loaded


## 6. REBEL Relation Extraction

In [10]:
# =============================================================================
# REBEL EXTRACTION
# =============================================================================

def parse_rebel_output(text):
    text = re.sub(r"<pad>", "", text)
    text = text.replace("<s>", "").replace("</s>", "")
    triples = []
    current = {"subject": "", "relation": "", "object": ""}
    state = None
    tokens = text.split()
    for token in tokens:
        if token == "<triplet>":
            if all(current.values()):
                triples.append((
                    current["subject"].strip(),
                    current["relation"].strip(),
                    current["object"].strip()
                ))
            current = {"subject": "", "relation": "", "object": ""}
            state = "subject"
            continue
        elif token == "<subj>":
            state = "object"
            continue
        elif token == "<obj>":
            state = "relation"
            continue
        if state:
            current[state] += token + " "
    if all(current.values()):
        triples.append((
            current["subject"].strip(),
            current["relation"].strip(),
            current["object"].strip()
        ))
    return triples


def rebel_batch(sentences):
    if not sentences:
        return []
    inputs = rebel_tokenizer(
        sentences,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = rebel_model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            return_dict_in_generate=True,
            output_scores=True
        )
    decoded = rebel_tokenizer.batch_decode(outputs.sequences, skip_special_tokens=False)
    results = []
    for text, score_tensor in zip(decoded, outputs.sequences_scores):
        beam_score = torch.sigmoid(score_tensor).item()
        results.append((text, beam_score))
    return results


def rebel_confidence(beam_score, subj_ground, obj_ground):
    ground_avg = (subj_ground + obj_ground) / 2
    confidence = (0.55 * beam_score + 0.45 * ground_avg)
    return round(confidence, 3)


def rebel_hallucination_filter(subj, rel, obj, sentence, ner_entities):
    subj = normalize_entity(subj)
    obj = normalize_entity(obj)
    rel = normalize_entity(rel)
    if rel in BAD_REBEL_RELATIONS:
        return False
    if subj == obj:
        return False
    if subj in BAD_PRONOUNS:
        return False
    if obj in BAD_PRONOUNS:
        return False
    subj_ground = grounding_score(subj, sentence, ner_entities)
    obj_ground = grounding_score(obj, sentence, ner_entities)
    if subj_ground < MIN_GROUND_SCORE:
        return False
    if obj_ground < MIN_GROUND_SCORE:
        return False
    return True


def rebel_extract_batch(sentence_batch, parsed_batch):
    generated = rebel_batch(sentence_batch)
    batch_results = []
    for idx, (sentence, (output, beam_score)) in enumerate(zip(sentence_batch, generated)):
        doc, ner_entities = parsed_batch[idx]
        parsed = parse_rebel_output(output)
        triples = []
        for subj, rel, obj in parsed:
            subj = clean_entity(subj)
            obj = clean_entity(obj)
            rel = normalize_entity(rel)
            if not subj or not obj:
                continue
            if not rebel_hallucination_filter(subj, rel, obj, sentence, ner_entities):
                continue
            subj_ground = grounding_score(subj, sentence, ner_entities)
            obj_ground = grounding_score(obj, sentence, ner_entities)
            conf = rebel_confidence(beam_score, subj_ground, obj_ground)
            if conf < MIN_REBEL_CONF:
                continue
            triples.append((subj, rel, obj, conf, "rebel"))
        batch_results.append(triples)
    return batch_results

print("REBEL extraction loaded")

REBEL extraction loaded


## 7. SpaCy Dependency Extraction

In [11]:
# =============================================================================
# SPACY DEPENDENCY EXTRACTION
# =============================================================================

def parse_sentence(sentence):
    doc = nlp(sentence)
    update_entity_memory(doc)
    ner_entities = set()
    for ent in doc.ents:
        if ent.label_ in VALID_NER:
            ner_entities.add(normalize_entity(ent.text))
    return doc, ner_entities


def subtree_text(token):
    text = " ".join(t.text for t in token.subtree)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def dependency_extract(doc, sentence, ner_entities):
    triples = []
    if noisy_sentence(sentence):
        return triples
    for verb in doc:
        if verb.pos_ not in {"VERB", "AUX"}:
            continue
        subj = None
        obj = None
        negated = False
        relation = verb.lemma_
        for child in verb.children:
            if child.dep_ == "neg":
                negated = True
        if negated:
            relation = "NOT_" + relation
        for child in verb.children:
            if child.dep_ in {"nsubj", "nsubjpass"}:
                subj = entity_span(child)
                break
        for child in verb.children:
            if child.dep_ in {"dobj", "attr", "oprd", "obj"}:
                obj = entity_span(child)
                break
        if obj is None:
            for child in verb.children:
                if child.dep_ == "prep":
                    for gc in child.children:
                        if gc.dep_ == "pobj":
                            obj = entity_span(gc)
                            break
                if obj:
                    break
        if not subj or not obj or subj == obj:
            continue
        if relation in BLACKLIST_RELATIONS:
            continue
        subj_score = grounding_score(subj, sentence, ner_entities)
        obj_score = grounding_score(obj, sentence, ner_entities)
        avg_ground = (subj_score + obj_score) / 2
        if avg_ground < MIN_GROUND_SCORE:
            continue
        confidence = (0.60 + avg_ground * 0.35)
        triples.append((subj, relation, obj, round(confidence, 3), "dependency"))
    return triples

print("Dependency extraction loaded")

Dependency extraction loaded


## 8. Process Each Part: NO Coreference + Relation Extraction

In [12]:
# =============================================================================
# BATCH PROCESSING (NO coreference)
# =============================================================================

def process_sentence_batch(sentence_batch):
    parsed_batch = [parse_sentence(s) for s in sentence_batch]
    rebel_results = rebel_extract_batch(sentence_batch, parsed_batch)
    batch_output = []
    for idx, (sentence, rebel_triples) in enumerate(zip(sentence_batch, rebel_results)):
        doc, ner_entities = parsed_batch[idx]
        dep_triples = dependency_extract(doc, sentence, ner_entities)
        combined = rebel_triples + dep_triples
        cleaned = [triple for triple in combined if validate_triple(triple)]
        batch_output.append(cleaned)
    return batch_output


def add_triple_to_graph(graph, triple, article):
    s, r, o, conf, src = triple
    s = resolve_pronoun(s)
    o = resolve_pronoun(o)
    s = canonicalize_entity(s)
    o = canonicalize_entity(o)
    s = normalize_entity(s)
    r = normalize_entity(r)
    o = normalize_entity(o)
    key = (s, r, o)

    if key in EDGE_STORE:
        edge_data = EDGE_STORE[key]
        edge_data["confidence"] = round(min(1.0, edge_data["confidence"] * 0.7 + conf * 0.3), 3)
        edge_data["mentions"] += 1
        old_sources = set(edge_data["sources"].split(",")) if edge_data["sources"] else set()
        old_sources.add(src)
        edge_data["sources"] = ",".join(sorted(old_sources))
        edge_data["articles"].add(article)
        return False

    EDGE_STORE[key] = {
        "confidence": conf,
        "sources": src,
        "mentions": 1,
        "articles": {article}
    }

    ENTITY_COUNTER[s] += 1
    ENTITY_COUNTER[o] += 1
    REL_COUNTER[r] += 1

    graph.add_node(s, label=s, type="entity")
    graph.add_node(o, label=o, type="entity")
    graph.add_edge(s, o, relation=r, confidence=conf, sources=src, mentions=1, article=article)
    return True


def process_part_no_coref(part_df, part_num):
    """
    Process one part of the data WITHOUT coreference resolution:
    1. Use original text directly (no coref)
    2. Extract relations using REBEL and SpaCy dependency parsing
    3. Group by article title
    """
    all_triples = []

    # Group by article title
    grouped = part_df.groupby('title')
    article_groups = [group for _, group in grouped]

    for group in tqdm(article_groups, desc=f"Processing Part {part_num}"):
        title = group['title'].iloc[0]
        text = " ".join(group['text'].astype(str))

        # Skip empty texts
        if not text or len(text.strip()) < 20:
            continue

        # NO coreference resolution - use original text directly
        original_text = text

        # Reset entity memory per article (no coref means pronouns are ambiguous across articles)
        global LAST_PERSON, LAST_ORG, LAST_GPE
        LAST_PERSON = None
        LAST_ORG = None
        LAST_GPE = None

        # Sentence splitting and batch processing
        doc = nlp(original_text)
        sentences = [
            s.text.strip()
            for s in doc.sents
            if len(s.text.strip()) > 10
        ]

        if not sentences:
            continue

        for i in range(0, len(sentences), BATCH_SIZE):
            sentence_batch = sentences[i:i + BATCH_SIZE]
            results = process_sentence_batch(sentence_batch)
            for sentence, triples in zip(sentence_batch, results):
                for triple in triples:
                    add_triple_to_graph(G, triple, title)
                    all_triples.append({
                        'article_title': title,
                        'subject': triple[0],
                        'relation': triple[1],
                        'object': triple[2],
                        'confidence': triple[3],
                        'source': triple[4],
                        'has_coref': False
                    })

    return all_triples

print("Process function defined (NO coreference)")

Process function defined (NO coreference)


In [17]:
# Process all 3 parts
all_extracted_triples_no_coref = []

for i, part in enumerate(parts):
    print(f"\n{'='*60}")
    print(f"Processing Part {i+1}/{len(parts)}")
    print(f"{'='*60}")
    
    part_triples = process_part_no_coref(part, i+1)
    all_extracted_triples_no_coref.extend(part_triples)
    
    # Save intermediate results
    with open(f'part_{i+1}_triples_no_coref.pkl', 'wb') as f:
        pickle.dump(part_triples, f)
    
    print(f"Part {i+1} complete: {len(part_triples)} triples extracted")

print(f"\nTotal triples extracted: {len(all_extracted_triples_no_coref)}")


Processing Part 1/3


Processing Part 1: 100%|██████████| 1450/1450 [3:45:30<00:00,  9.33s/it]  


Part 1 complete: 39154 triples extracted

Processing Part 2/3


Processing Part 2: 100%|██████████| 1450/1450 [3:34:02<00:00,  8.86s/it]  


Part 2 complete: 31790 triples extracted

Processing Part 3/3


Processing Part 3: 100%|██████████| 1452/1452 [4:26:23<00:00, 11.01s/it]  

Part 3 complete: 28567 triples extracted

Total triples extracted: 99511


## 9. Finalize and Save Graph

In [18]:
# =============================================================================
# FINALIZE GRAPH
# =============================================================================

def finalize_graph(graph):
    for (s, r, o), meta in EDGE_STORE.items():
        # Update edge attributes with merged metadata
        if graph.has_edge(s, o):
            for edge_key in graph[s][o]:
                edge_data = graph[s][o][edge_key]
                if edge_data.get("relation") == r:
                    edge_data["confidence"] = meta["confidence"]
                    edge_data["mentions"] = meta["mentions"]
                    edge_data["sources"] = meta["sources"]
                    edge_data["articles"] = "; ".join(sorted(meta["articles"]))

finalize_graph(G)

print(f"Graph finalized: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph finalized: 110921 nodes, 96236 edges


In [19]:
# =============================================================================
# SAVE OUTPUTS
# =============================================================================

# Save as GraphML
nx.write_graphml(G, "knowledge_graph_no_coref.graphml")
print("Saved: knowledge_graph_no_coref.graphml")

# Save as pickle
with open("knowledge_graph_no_coref.pkl", "wb") as f:
    pickle.dump(G, f)
print("Saved: knowledge_graph_no_coref.pkl")

# Save triples as JSON
with open("triples_no_coref.json", "w", encoding="utf-8") as f:
    json.dump(all_extracted_triples_no_coref, f, indent=2, ensure_ascii=False)
print("Saved: triples_no_coref.json")

# Save triples as CSV for easy inspection
triples_df_no_coref = pd.DataFrame(all_extracted_triples_no_coref)
triples_df_no_coref.to_csv("triples_no_coref.csv", index=False)
print("Saved: triples_no_coref.csv")

# Save graph state
with open("graph_no_coref_state.pkl", "wb") as f:
    pickle.dump({
        "graph": G,
        "edge_store": dict(EDGE_STORE),
        "rel_counter": dict(REL_COUNTER),
        "entity_counter": dict(ENTITY_COUNTER)
    }, f)
print("Saved: graph_no_coref_state.pkl")

print("\nAll files saved successfully!")

Saved: knowledge_graph_no_coref.graphml
Saved: knowledge_graph_no_coref.pkl
Saved: triples_no_coref.json
Saved: triples_no_coref.csv
Saved: graph_no_coref_state.pkl

All files saved successfully!


## 10. Graph Analysis

In [20]:
# =============================================================================
# GRAPH ANALYSIS
# =============================================================================

print("=== Knowledge Graph WITHOUT Coreference Resolution ===\n")
print(f"Total Nodes: {G.number_of_nodes()}")
print(f"Total Edges (triples): {G.number_of_edges()}")
print(f"Average Degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")

# Connected components
connected_components = list(nx.connected_components(G))
print(f"\nConnected Components: {len(connected_components)}")
print(f"Largest Component Size: {len(max(connected_components, key=len))} nodes")

# Most connected nodes
degree_centrality = nx.degree_centrality(G)
top_nodes = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 Most Connected Nodes:")
for node, centrality in top_nodes:
    print(f"  {node}: {centrality:.4f}")

# Source distribution
source_counts = triples_df_no_coref['source'].value_counts()
print("\nTriple Sources:")
print(source_counts)

=== Knowledge Graph WITHOUT Coreference Resolution ===

Total Nodes: 110921
Total Edges (triples): 96236
Average Degree: 1.74

Connected Components: 26605
Largest Component Size: 47488 nodes

Top 10 Most Connected Nodes:
  which: 0.0161
  that: 0.0091
  the united states: 0.0068
  who dey: 0.0050
  world war ii: 0.0028
  the world health organization who: 0.0024
  what: 0.0022
  new york: 0.0019
  new york city: 0.0014
  the soviet union: 0.0014

Triple Sources:
source
dependency    71350
rebel         28161
Name: count, dtype: int64


## 11. Inspect Sample Triples

In [21]:
# Show sample triples
print("Sample Triples (without coreference resolution):\n")
for i, triple in enumerate(all_extracted_triples_no_coref[:20]):
    print(f"{i+1}. [{triple['source']}] {triple['subject']} --[{triple['relation']}]--> {triple['object']} (conf={triple['confidence']}, Article: {triple['article_title']})")

Sample Triples (without coreference resolution):

1. [dependency] lynne spears --[negotiate]--> her manager (conf=0.836, Article: "...Baby One More Time (album)")
2. [dependency] innosense --[ask]--> a spears family friend and entertainment lawyer larry rudolph (conf=0.836, Article: "...Baby One More Time (album)")
3. [rebel] eric foster white --[record label]--> jive records (conf=0.724, Article: "...Baby One More Time (album)")
4. [rebel] max martin --[country of citizenship]--> sweden (conf=0.718, Article: "...Baby One More Time (album)")
5. [rebel] denniz pop --[country of citizenship]--> sweden (conf=0.718, Article: "...Baby One More Time (album)")
6. [rebel] rami yacoub --[country of citizenship]--> sweden (conf=0.718, Article: "...Baby One More Time (album)")
7. [dependency] lynne spears --[travel]--> sweden (conf=0.784, Article: "...Baby One More Time (album)")
8. [dependency] lynne spears --[ask]--> larry rudolph (conf=0.792, Article: "...Baby One More Time (album)")
9. [rebel